# Matryoshka Finetuning

In [1]:
import sys, os, yaml
import pandas as pd
from pathlib import Path
import torch, pickle

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.esci.data import sample_dataset
from src.esci.matryoshka_train import train_matryoshka
from src.esci.system_a import encode_systemA
from src.esci.system_b import build_candidates
from src.esci.metrics import compute_recall_metrics, compute_ndcg_metrics
from src.esci.sparse_retrievers import SPLADEFast, BM25Fast

# Load Config
with open(project_root / "configs" / "esci.yaml") as f:
    cfg = yaml.safe_load(f)

# Load Data
print("Loading Processed Data...")
pair_df = pd.read_parquet(cfg["paths"]["processed_dir"] + "/pair_df.parquet")

# Build Ground Truth (Critical for Metrics)
print("Building Ground Truth Map (q_rels)...")
q_rels = pair_df.groupby("query_id")["grade"].apply(list).to_dict()

Loading Processed Data...
Building Ground Truth Map (q_rels)...


### Training Execution

In [2]:
# TRAINING ARCHITECTURE: MRL + MNRL
# 1. Loss Function: MultipleNegativesRankingLoss (MNRL)
#    We use in-batch negatives. Every other sample in the batch (size 64) 
#    acts as a negative for the current anchor, providing a strong training signal 
#
# 2. Matryoshka Adaptation
#    The model is forced to learn a hierarchical representation where the 
#    first 64 dimensions carry the vast majority of the semantic signal.

train_matryoshka(pair_df, cfg)

 Loading Backbone: BAAI/bge-base-en-v1.5
 Config: Batch=64 | Epochs=5 | LR=2e-05
 Max Seq Length: 256
--> Filtering data for Training...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

--> Starting Training...


C:\Users\aminl\anaconda3\envs\pytorch-gpu\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
500,4.354200
1000,2.599500
1500,2.105300
2000,1.855700
2500,1.625000
3000,1.481100
3500,1.329300
4000,1.233200
4500,1.133500
5000,1.123500


C:\Users\aminl\anaconda3\envs\pytorch-gpu\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\aminl\anaconda3\envs\pytorch-gpu\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  retur

#### Load Baseline Results (from Notebook 02)

In [3]:
baseline_df = pd.read_csv("../results/baseline_metrics_per_dim.csv")
print(f"Loaded Baseline Results: {len(baseline_df)} rows")

Loaded Baseline Results: 20 rows


### Encoding With Matryoshka Model

In [4]:
matryoshka_path = str(Path(cfg["paths"]["matryoshka_dir"]) / "us")
print(f"Evaluating Matryoshka Finetuned Model: {matryoshka_path}")
encode_systemA(pair_df, cfg, model_override=matryoshka_path)

Evaluating Matryoshka Finetuned Model: ..\artifacts\matryoshka_models\us
 Encoding with System A model: ..\artifacts\matryoshka_models\us
 Max Seq Length: 256
    -> Preparing Product List...
    -> Encoding 164001 Products...


Batches:   0%|          | 0/5126 [00:00<?, ?it/s]

    -> Preparing Query List...
    Applying BGE instruction prefix: 'Represent this sentence for searching relevant passages: '
    -> Encoding 8908 Queries...


Batches:   0%|          | 0/279 [00:00<?, ?it/s]

 System A Encoding Complete.


### Loading Sparse Artifacts

In [5]:
print("Loading SPLADE Matrix...")
splade_data = torch.load("../artifacts/systemA/splade_data.pt", map_location="cpu")

splade_loaded = SPLADEFast(
    cfg["sparse"]["splade_model"], 
    batch_size=cfg["sparse"]["batch_size"]
)

splade_loaded.doc_matrix = splade_data["doc_matrix"]
splade_loaded.pid_list   = splade_data["pid_list"]

# CRITICAL: device alignment
splade_loaded.doc_matrix = splade_loaded.doc_matrix.to(splade_loaded.device)

# Safety
assert splade_loaded.doc_matrix.shape[0] == len(splade_loaded.pid_list)

print("Loading BM25 Index...")
with open("../artifacts/systemA/bm25_retriever.pkl", "rb") as f:
    bm25_loaded = pickle.load(f)

splade = splade_loaded
bm25 = bm25_loaded

Loading SPLADE Matrix...


C:\Users\aminl\anaconda3\envs\pytorch-gpu\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loading BM25 Index...


## Running Experiment Loop

In [7]:
results = []
dims = cfg["matryoshka"]["dims"]
strategies = {
    "Dense Only": ["dense"],
    "Dense + BM25": ["dense", "bm25"],
    "Dense + SPLADE": ["dense", "splade"],
    "Dense + BM25 + SPLADE": ["dense", "bm25", "splade"]
}

max_len = cfg["matryoshka"]["max_seq_length"]

for strat, sources in strategies.items():
    print(f"\n Strategy: {strat}")
    cfg["retrieval"]["sources"] = sources
    
    for dim in dims:
        df, qps = build_candidates(
            cfg, split="test", override_dim=dim,
            prebuilt_splade=splade if "splade" in sources else None,
            prebuilt_bm25=bm25 if "bm25" in sources else None
        )
        
        # Pass q_rels here!
        # Compute them separately
        recall_results = compute_recall_metrics(df, q_rels)
        ndcg_results = compute_ndcg_metrics(df, q_rels)
        
        # Merge them into one final dictionary
        final_metrics = {**recall_results, **ndcg_results}
        
        results.append({
            "Model": "Matryoska",
            "Strategy": strat,
            "Dimension": dim,
            "Max Length": max_len,
            "Recall@50": final_metrics["Recall@50"],
            "Recall@100": final_metrics["Recall@100"],
            "Recall@200": final_metrics["Recall@200"],
            "nDCG@10": final_metrics["nDCG@10"],
            "nDCG@20": final_metrics["nDCG@20"],
            "nDCG@50": final_metrics["nDCG@50"],
            "QPS": round(qps, 2)
        })


 Strategy: Dense Only
 Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=64)
    -> Measuring QPS for 8908 queries...

 Strategy: Dense + BM25
 Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=64)
    -> Measuring QPS for 8908 queries...

 Strategy: Dense + SPLADE
 Using FAISS (Dim=768)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=512)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=256)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=128)
    -> Measuring QPS for 8908 queries...
 Using FAISS (Dim=64)
    -> Me

We compare the Recall@200 of the Fine-Tuned model against the Baseline
specifically at the 64-dimension truncation.
Success is defined as: Fine-Tuned@64 > Baseline@64.

In [9]:
# Save Matryoshka results
matryoshka_df = pd.DataFrame(results)
matryoshka_df.to_csv("../results/matryoshka_metrics_per_dim.csv", index=False)
print("Saved Matryoshka metrics to ../results/matryoshka_metrics_per_dim.csv")

Saved Matryoshka metrics to ../results/matryoshka_metrics_per_dim.csv


### Comparing results

In [10]:
# Filter for the primary metrics and keys
cols_to_keep = ['Strategy', 'Dimension', 'Recall@200', 'nDCG@20']
baseline_sub = baseline_df[cols_to_keep]
matryoshka_sub = matryoshka_df[cols_to_keep]

# Merge to compare side-by-side
comparison = pd.merge(
    baseline_sub, 
    matryoshka_sub, 
    on=['Strategy', 'Dimension'], 
    suffixes=('_Base', '_Matryoshka')
)

# Calculate Lifts
comparison['Recall_Lift (%)'] = (
    (comparison['Recall@200_Matryoshka'] - comparison['Recall@200_Base']) / comparison['Recall@200_Base'] * 100
)
comparison['nDCG_Lift (%)'] = (
    (comparison['nDCG@20_Matryoshka'] - comparison['nDCG@20_Base']) / comparison['nDCG@20_Base'] * 100
)

# Display the 64-dimension results (The Matryoshka target)
print("Performance Comparison at 64 Dimensions:")
display_cols = [
    'Strategy', 'Recall@200_Base', 'Recall@200_Matryoshka', 
    'Recall_Lift (%)', 'nDCG@20_Base', 'nDCG@20_Matryoshka', 'nDCG_Lift (%)'
]
display(comparison[comparison['Dimension'] == 64][display_cols].round(2))

Performance Comparison at 64 Dimensions:


,Strategy,Recall@200_Base,Recall@200_Matryoshka,Recall_Lift (%),nDCG@20_Base,nDCG@20_Matryoshka,nDCG_Lift (%)
4,Dense Only,0.43,0.74,73.18,0.25,0.41,63.18
9,Dense + BM25,0.56,0.78,40.39,0.32,0.44,39.83
14,Dense + SPLADE,0.66,0.81,22.12,0.35,0.47,33.07
19,Dense + BM25 + SPLADE,0.70,0.81,16.01,0.39,0.48,24.33


The results shows that the finetuned model (Dense @64) was successful at:
 - truncating the baseline vectors to 64 dimensions.
 - outperforming the baseline model (Dense @64) Recall@200: 0.74 vs 0.43. (which is basically the same results as baseline@768)
 The use of Multiple Negatives Ranking Loss (MNRL) was the key of these performances.

For the next step, the logical choice is to stick with Dense+BM25+SPLADE @64 for its performance and acceptable latency.

In [11]:
matryoshka_df

,Model,Strategy,Dimension,Max Length,Recall@50,Recall@100,Recall@200,nDCG@10,nDCG@20,nDCG@50,QPS
0,Matryoska,Dense Only,768,256,0.594722,0.696971,0.777085,0.415561,0.451197,0.509902,42.29
1,Matryoska,Dense Only,512,256,0.591238,0.693865,0.774935,0.411647,0.447176,0.505781,62.87
2,Matryoska,Dense Only,256,256,0.583986,0.686483,0.767819,0.405056,0.439302,0.498136,132.73
3,Matryoska,Dense Only,128,256,0.570477,0.673743,0.757811,0.392283,0.426478,0.485493,245.47
4,Matryoska,Dense Only,64,256,0.548519,0.652984,0.739383,0.373069,0.406642,0.464584,446.93
5,Matryoska,Dense + BM25,768,256,0.614730,0.719386,0.805419,0.435294,0.472049,0.531403,34.29
6,Matryoska,Dense + BM25,512,256,0.612490,0.716557,0.803955,0.432421,0.469512,0.528392,50.51
7,Matryoska,Dense + BM25,256,256,0.606458,0.711954,0.800024,0.427862,0.463763,0.522964,85.69
8,Matryoska,Dense + BM25,128,256,0.596966,0.702358,0.792946,0.420100,0.454703,0.514266,121.17
9,Matryoska,Dense + BM25,64,256,0.578072,0.687231,0.782572,0.407317,0.440856,0.499068,161.35


In [12]:
baseline_df

,Model,Strategy,Dimension,Max Length,Recall@50,Recall@100,Recall@200,nDCG@10,nDCG@20,nDCG@50,QPS
0,Baseline,Dense Only,768,256,0.569735,0.662507,0.739768,0.433009,0.463072,0.515612,41.90
1,Baseline,Dense Only,512,256,0.555104,0.647921,0.724900,0.422693,0.451474,0.503189,68.60
2,Baseline,Dense Only,256,256,0.511757,0.601916,0.681114,0.395021,0.420819,0.469609,131.59
3,Baseline,Dense Only,128,256,0.430857,0.512219,0.588258,0.338874,0.360226,0.403421,244.61
4,Baseline,Dense Only,64,256,0.290365,0.357380,0.426956,0.235852,0.249192,0.281322,446.34
5,Baseline,Dense + BM25,768,256,0.592636,0.685852,0.762853,0.448713,0.481051,0.534117,36.39
6,Baseline,Dense + BM25,512,256,0.580858,0.674523,0.752546,0.441316,0.472488,0.524907,51.92
7,Baseline,Dense + BM25,256,256,0.544849,0.639521,0.721953,0.423910,0.451498,0.500605,81.65
8,Baseline,Dense + BM25,128,256,0.474590,0.566390,0.659875,0.386450,0.406928,0.450027,111.40
9,Baseline,Dense + BM25,64,256,0.344066,0.431098,0.557411,0.306621,0.315276,0.348642,180.56
